# ACE++ GRPO Training Notebook

This notebook is a minimal, GPU-ready RL training workflow for ACE++ using:
- `trl.GRPOTrainer`
- `unsloth.FastLanguageModel`
- the OpenEnv-compatible ACE++ multi-agent wrapper

The trained policy controls one agent inside a hidden-state economic round while scripted peer agents create coalition, competition, trust, and tool-use pressure. It is optimized for correctness, clarity, and observable learning rather than absolute benchmark performance.

Load Model with Unsloth

In [ ]:
import importlib
import json
import os
import random
import sys
import warnings
from pathlib import Path


def require_notebook_dependencies():
    missing = []
    modules = {}
    for import_name, package_name in [
        ("matplotlib.pyplot", "matplotlib"),
        ("torch", "torch"),
        ("datasets", "datasets"),
        ("transformers", "transformers"),
        ("trl", "trl"),
        ("unsloth", "unsloth"),
    ]:
        try:
            modules[import_name] = importlib.import_module(import_name)
        except ModuleNotFoundError:
            missing.append(package_name)

    if missing:
        raise RuntimeError(
            "Missing notebook dependencies: "
            + ", ".join(sorted(set(missing)))
            + ". Run the install cell first, then restart the runtime if needed."
        )
    return modules


_modules = require_notebook_dependencies()
plt = _modules["matplotlib.pyplot"]
torch = _modules["torch"]
Dataset = _modules["datasets"].Dataset
set_seed = _modules["transformers"].set_seed
TrainerCallback = _modules["transformers"].TrainerCallback
trl_module = _modules["trl"]
GRPOTrainer = getattr(trl_module, "GRPOTrainer", None)
GRPOConfig = getattr(trl_module, "GRPOConfig", None)
if GRPOConfig is None:
    try:
        GRPOConfig = importlib.import_module("trl.trainer.grpo_config").GRPOConfig
    except (ModuleNotFoundError, AttributeError) as exc:
        version = getattr(trl_module, "__version__", "unknown")
        raise RuntimeError(
            f"Installed TRL version ({version}) does not expose GRPOConfig. "
            "Run the install cell with trl>=0.17.0, then restart the runtime/kernel."
        ) from exc
if GRPOTrainer is None:
    try:
        GRPOTrainer = importlib.import_module("trl.trainer.grpo_trainer").GRPOTrainer
    except (ModuleNotFoundError, AttributeError) as exc:
        version = getattr(trl_module, "__version__", "unknown")
        raise RuntimeError(
            f"Installed TRL version ({version}) does not expose GRPOTrainer. "
            "Run the install cell with trl>=0.17.0, then restart the runtime/kernel."
        ) from exc
FastLanguageModel = _modules["unsloth"].FastLanguageModel

set_seed(42)
random.seed(42)
warnings.filterwarnings("ignore", message="Both `max_new_tokens`.*")
warnings.filterwarnings("ignore", message="`use_return_dict` is deprecated.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")

assert torch.cuda.is_available(), "This notebook is intended for a GPU runtime. Use a Colab/HF GPU runtime."

# Default is T4-friendly. For a costly A100 run, you can switch to:
# MODEL_NAME = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.config.use_cache = False
model.generation_config.max_length = None
print("Loaded model:", MODEL_NAME)

## Cell 3 — Import Environment and Build Env Reward Wrapper

In [ ]:
# The notebook should be run from the repository root. This also helps Colab
# after uploading/cloning the repo because the current directory is added first.
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from ace_env_openenv import ACEOpenEnv, ACEOpenMultiAgentEnv
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Could not import ACE++ environment code. Run this notebook from the repo root "
        "or clone/upload the repository before running training cells."
    ) from exc

ROUND_TYPES = ["cooperative", "competitive", "resource"]
ECONOMIC_TOOLS = {"submit_bid", "allocate_resources", "execute_contract"}
COALITION_TOOLS = {"propose_alliance", "accept_alliance", "reject_alliance", "betray", "challenge"}
VALID_TOOLS = ECONOMIC_TOOLS | COALITION_TOOLS


def extract_first_json_object(text: str):
    start = text.find("{")
    if start == -1:
        return None
    decoder = json.JSONDecoder()
    for idx in range(start, len(text)):
        if text[idx] != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[idx:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


def safe_parse_completion(text: str):
    obj = extract_first_json_object(text)
    valid = True

    if not isinstance(obj, dict):
        valid = False
        obj = {}

    belief = obj.get("belief") if isinstance(obj.get("belief"), dict) else {}
    predicted_round = obj.get("predicted_round") or belief.get("predicted_round")
    if predicted_round not in ROUND_TYPES:
        valid = False
        predicted_round = "resource"

    action_field = obj.get("action")
    parameters = obj.get("parameters") if isinstance(obj.get("parameters"), dict) else {}

    if isinstance(action_field, dict):
        action = action_field.get("tool") or action_field.get("name")
        parameters = action_field.get("parameters") if isinstance(action_field.get("parameters"), dict) else parameters
    else:
        action = action_field

    if action not in VALID_TOOLS:
        valid = False
        action = "allocate_resources" if predicted_round == "resource" else "submit_bid"

    normalized_params = dict(parameters) if isinstance(parameters, dict) else {}
    if action in {"submit_bid", "allocate_resources"}:
        try:
            normalized_params["amount"] = float(normalized_params.get("amount", 50))
        except Exception:
            valid = False
            normalized_params["amount"] = 50.0
    elif action == "execute_contract":
        normalized_params.setdefault("team_id", None)
    elif action in {"propose_alliance", "reject_alliance", "challenge"}:
        try:
            normalized_params["target_id"] = int(normalized_params.get("target_id", normalized_params.get("partner_id", 1)))
        except Exception:
            valid = False
            normalized_params["target_id"] = 1
    elif action == "accept_alliance":
        try:
            normalized_params["proposer_id"] = int(normalized_params.get("proposer_id", normalized_params.get("target_id", 1)))
        except Exception:
            valid = False
            normalized_params["proposer_id"] = 1
    elif action == "betray":
        try:
            normalized_params["partner_id"] = int(normalized_params.get("partner_id", normalized_params.get("target_id", 1)))
        except Exception:
            valid = False
            normalized_params["partner_id"] = 1

    return {
        "predicted_round": predicted_round,
        "action": action,
        "parameters": normalized_params,
    }, valid


def build_single_round_env(ground_truth: str, payoff_seed: int, difficulty: str = "easy"):
    env = ACEOpenEnv(
        num_rounds=1,
        seed=0,
        round_type_schedule=[ground_truth],
        difficulty=difficulty,
    )
    env.reset()
    inner = getattr(env, "_env", env)
    inner.current_round_type = ground_truth
    inner.current_payoff_seed = int(payoff_seed)
    inner.current_payoff = inner._payoff_from_seed(int(payoff_seed))
    return env


def build_multiagent_round_env(ground_truth: str, payoff_seed: int, difficulty: str = "easy", num_agents: int = 3):
    env = ACEOpenMultiAgentEnv(
        num_agents=num_agents,
        num_rounds=1,
        seed=0,
        round_type_schedule=[ground_truth],
        difficulty=difficulty,
        id_shuffle=False,
        god_mode=True,
    )
    env.reset()
    core = getattr(getattr(env, "_env", env), "_core", None)
    if core is not None:
        core.current_round_type = ground_truth
        core.current_payoff_seed = int(payoff_seed)
        core.current_payoff = core._payoff_from_seed(int(payoff_seed))
        wrapped = getattr(env, "_env", env)
        wrapped.current_round_type = core.current_round_type
        wrapped.current_payoff_seed = core.current_payoff_seed
        wrapped.current_payoff = core.current_payoff
    return env


def scripted_peer_action(agent_id: int, ground_truth: str):
    partner_id = 1 if agent_id == 0 else 0
    if ground_truth == "competitive":
        tool, params = "challenge", {"target_id": partner_id}
    elif ground_truth == "cooperative":
        tool, params = "submit_bid", {"amount": 30, "partner_id": partner_id}
    else:
        tool, params = "allocate_resources", {"amount": 50}
    return json.dumps({
        "belief": {"predicted_round": ground_truth, "confidence": 0.78},
        "action": {"tool": tool, "parameters": params},
    })


def anti_bid_hack_penalty(parsed: dict, ground_truth: str) -> float:
    """Discourage one-size-fits-all extreme bidding while preserving valid strategic bids."""
    if parsed.get("action") != "submit_bid":
        return 0.0
    amount = float(parsed.get("parameters", {}).get("amount", 50.0))
    if amount < 0.0 or amount > 100.0:
        return -0.75
    if ground_truth == "cooperative" and amount > 45.0:
        return -0.45
    if ground_truth == "competitive" and amount < 60.0:
        return -0.35
    if ground_truth == "resource" and not (35.0 <= amount <= 65.0):
        return -0.35
    return 0.0


def run_env_step(prompt_output: str, ground_truth: str, payoff_seed: int, difficulty: str = "easy", agent_id: int = 0, num_agents: int = 3):
    env = build_multiagent_round_env(ground_truth, payoff_seed, difficulty=difficulty, num_agents=num_agents)
    parsed, valid = safe_parse_completion(prompt_output)

    model_action = json.dumps(
        {
            "belief": {
                "predicted_round": parsed["predicted_round"],
                "confidence": 0.8 if valid else 0.2,
            },
            "action": {
                "tool": parsed["action"],
                "parameters": parsed["parameters"],
            },
        }
    )

    actions = [scripted_peer_action(i, ground_truth) for i in range(num_agents)]
    actions[int(agent_id)] = model_action
    _, rewards, _, info = env.step(actions)

    # Make world-modeling the primary objective. The env reward is useful, but
    # clipped so action magnitude cannot dominate hidden-state inference.
    correct = parsed["predicted_round"] == ground_truth
    inference_reward = 2.0 if correct else -1.0
    env_reward = max(-1.0, min(1.0, float(rewards[int(agent_id)])))
    bid_penalty = anti_bid_hack_penalty(parsed, ground_truth)
    # Explicit format + concision signals keep early RL from learning long rambling completions.
    compact_json_bonus = 0.15 if valid and len(prompt_output) <= 260 else 0.0
    runaway_penalty = -0.25 if len(prompt_output) > 520 else 0.0
    format_reward = 0.25 if valid else -0.50
    reward = inference_reward + 0.35 * env_reward + format_reward + compact_json_bonus + runaway_penalty + bid_penalty
    reward_breakdown = info.get("step_log", {}).get("reward_breakdown", [{}])[int(agent_id)]
    reward_breakdown = {
        **reward_breakdown,
        "r_inference_explicit": inference_reward,
        "r_env_clipped": env_reward,
        "r_format": format_reward,
        "r_bid_hack_penalty": bid_penalty,
        "r_compact_json": compact_json_bonus,
        "r_runaway": runaway_penalty,
    }

    return {
        "reward": reward,
        "correct": correct,
        "valid": valid,
        "info": info,
        "parsed": parsed,
        "reward_breakdown": reward_breakdown,
    }

## Cell 4 — Define Prompt Format

In [ ]:
SYSTEM_INSTRUCTION = """You are one company-agent inside ACE++, a partially observable multi-agent economy.
Infer the hidden market round from noisy numeric market signals, peer context, trust, and alliances. Then choose one valid tool action.
Direct round labels and direct cooperation/competition labels are intentionally hidden.

Hidden round types:
- cooperative: coalitions and low bids are rewarded
- competitive: direct challenges or aggressive solo bids are rewarded
- resource: balanced allocation and supply protection are rewarded

Available tools:
- submit_bid with parameters {\"amount\": float, optional \"partner_id\": int}
- allocate_resources with parameters {\"amount\": float}
- execute_contract with parameters {\"team_id\": int|null}
- propose_alliance / reject_alliance / challenge with parameters {\"target_id\": int}
- accept_alliance with parameters {\"proposer_id\": int}
- betray with parameters {\"partner_id\": int}

Output ONLY valid JSON in exactly this format:
{
  \"predicted_round\": \"cooperative|competitive|resource\",
  \"action\": \"submit_bid|allocate_resources|execute_contract|propose_alliance|accept_alliance|reject_alliance|betray|challenge\",
  \"parameters\": {\"amount\": 50}
}

Important: return one compact JSON object only. No markdown, no reasoning, no repeated text. Stop immediately after the closing brace.
"""


def public_market_state_for_training(market_state: dict):
    """Hide direct label-like fields so training requires inference from noisy signals."""
    return {
        "demand_index": market_state.get("demand_index"),
        "volatility": market_state.get("volatility"),
    }


def format_prompt(observation: dict, agent_id: int = 0):
    history = observation.get("history", [])
    user_content = "\n".join(
        [
            f"You control agent_id={agent_id}.",
            f"Visible public agent IDs: {observation.get('public_ids', [0, 1, 2])}",
            "",
            "Recent history:",
            json.dumps(history, indent=2),
            "",
            "Current market_state:",
            json.dumps(public_market_state_for_training(observation["market_state"]), indent=2),
            "",
            "Current trust summary:",
            json.dumps(observation.get("trust", {}), indent=2),
            "",
            "Current alliances:",
            json.dumps(observation.get("alliances", []), indent=2),
            "",
            "Return ONLY the compact JSON object. Do not write 'Human', markdown, or explanation.",
        ]
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return SYSTEM_INSTRUCTION + "\n\n" + user_content + "\nAssistant JSON:\n"


def build_prompt_dataset(n_samples: int = 64, difficulty="medium", seed: int = 123, num_agents: int = 3):
    rows = []
    difficulties = difficulty if isinstance(difficulty, list) else [difficulty]
    round_schedule = [ROUND_TYPES[i % len(ROUND_TYPES)] for i in range(n_samples)]
    rng = random.Random(seed)
    rng.shuffle(round_schedule)
    for i in range(n_samples):
        current_difficulty = difficulties[i % len(difficulties)]
        scheduled_round = round_schedule[i]
        env = ACEOpenMultiAgentEnv(
            num_agents=num_agents,
            num_rounds=1,
            seed=seed + i,
            round_type_schedule=[scheduled_round],
            difficulty=current_difficulty,
            id_shuffle=False,
            god_mode=True,
        )
        observation = env.reset()
        state = env.state()
        agent_id = i % num_agents
        rows.append(
            {
                "prompt": format_prompt(observation, agent_id=agent_id),
                "ground_truth": state["current_round_type"],
                "payoff_seed": int(state["current_payoff_seed"]),
                "difficulty": current_difficulty,
                "agent_id": agent_id,
                "num_agents": num_agents,
            }
        )
    return Dataset.from_list(rows)


train_dataset = build_prompt_dataset(n_samples=360, difficulty=["medium", "hard"], seed=123)
eval_dataset = build_prompt_dataset(n_samples=60, difficulty="hard", seed=999)
train_dataset[0]

## Cell 5 — Rollout and Reward Functions

In [ ]:
reward_log = []
heldout_eval_log = []


@torch.no_grad()
def generate_completions(prompts, model, tokenizer, max_new_tokens: int = 64, temperature: float = 0.0):
    model.eval()
    tokenized = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.95})
    else:
        generation_kwargs.update({"do_sample": False})

    outputs = model.generate(**tokenized, **generation_kwargs)

    completions = []
    prompt_length = tokenized["input_ids"].shape[1]
    for row_idx in range(outputs.shape[0]):
        completion_ids = outputs[row_idx, prompt_length:]
        completions.append(tokenizer.decode(completion_ids, skip_special_tokens=True).strip())
    return completions


def generate_and_score(prompts, rows, model=model, tokenizer=tokenizer, max_new_tokens: int = 64, temperature: float = 0.0):
    completions = generate_completions(prompts, model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens, temperature=temperature)
    rewards = []
    results = []
    for completion, row in zip(completions, rows):
        result = run_env_step(
            completion,
            row["ground_truth"],
            row["payoff_seed"],
            row["difficulty"],
            agent_id=row.get("agent_id", 0),
            num_agents=row.get("num_agents", 3),
        )
        rewards.append(result["reward"])
        results.append(result)
    return completions, rewards, results


def reward_func(prompts, completions, ground_truth, payoff_seed, difficulty, agent_id=None, num_agents=None, trainer_state=None, **kwargs):
    rewards = []
    correct = 0
    valid = 0
    explicit_inference_rewards = []
    clipped_env_rewards = []
    bid_hack_penalties = []

    if agent_id is None:
        agent_id = [0] * len(completions)
    if num_agents is None:
        num_agents = [3] * len(completions)

    for completion, gt, ps, diff, aid, nagents in zip(completions, ground_truth, payoff_seed, difficulty, agent_id, num_agents):
        result = run_env_step(completion, gt, ps, diff, agent_id=aid, num_agents=nagents)
        rewards.append(result["reward"])
        correct += int(result["correct"])
        valid += int(result["valid"])
        breakdown = result.get("reward_breakdown", {})
        explicit_inference_rewards.append(float(breakdown.get("r_inference_explicit", 0.0)))
        clipped_env_rewards.append(float(breakdown.get("r_env_clipped", 0.0)))
        bid_hack_penalties.append(float(breakdown.get("r_bid_hack_penalty", 0.0)))

    reward_log.append(
        {
            "step": getattr(trainer_state, "global_step", len(reward_log)),
            "avg_reward": sum(rewards) / max(1, len(rewards)),
            "accuracy": correct / max(1, len(rewards)),
            "invalid_rate": 1.0 - (valid / max(1, len(rewards))),
            "explicit_inference_reward": sum(explicit_inference_rewards) / max(1, len(explicit_inference_rewards)),
            "clipped_env_reward": sum(clipped_env_rewards) / max(1, len(clipped_env_rewards)),
            "bid_hack_penalty": sum(bid_hack_penalties) / max(1, len(bid_hack_penalties)),
        }
    )
    return rewards


def evaluate_model(model, tokenizer, dataset, n_rollouts: int = 24):
    rows = [dataset[i] for i in range(min(n_rollouts, len(dataset)))]
    prompts = [row["prompt"] for row in rows]
    completions, rewards, results = generate_and_score(prompts, rows, model=model, tokenizer=tokenizer, max_new_tokens=64, temperature=0.0)

    metrics = {
        "mean_reward": sum(rewards) / max(1, len(rewards)),
        "accuracy": sum(int(r["correct"]) for r in results) / max(1, len(results)),
        "valid_rate": sum(int(r["valid"]) for r in results) / max(1, len(results)),
        "prediction_counts": {rt: sum(1 for r in results if r["parsed"]["predicted_round"] == rt) for rt in ROUND_TYPES},
        "mode_collapse_rate": max(
            [sum(1 for r in results if r["parsed"]["predicted_round"] == rt) for rt in ROUND_TYPES]
        ) / max(1, len(results)),
        "samples": [
            {
                "completion": completion,
                "reward": reward,
                "parsed": result["parsed"],
                "correct": result["correct"],
                "valid": result["valid"],
                "reward_breakdown": result.get("reward_breakdown", {}),
            }
            for completion, reward, result in zip(completions[:3], rewards[:3], results[:3])
        ],
    }
    return metrics


class HeldoutEvalCallback(TrainerCallback):
    def __init__(self, every_steps: int = 50, n_rollouts: int = 24):
        self.every_steps = every_steps
        self.n_rollouts = n_rollouts

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if model is None or state.global_step == 0 or state.global_step % self.every_steps != 0:
            return control
        metrics = evaluate_model(model, tokenizer, eval_dataset, n_rollouts=self.n_rollouts)
        heldout_eval_log.append(
            {
                "step": int(state.global_step),
                "mean_reward": metrics["mean_reward"],
                "accuracy": metrics["accuracy"],
                "valid_rate": metrics["valid_rate"],
                "mode_collapse_rate": metrics["mode_collapse_rate"],
                "prediction_counts": metrics["prediction_counts"],
            }
        )
        model.train()
        print(
            f"Heldout eval step {state.global_step}: "
            f"acc={metrics['accuracy']:.3f}, reward={metrics['mean_reward']:.3f}, "
            f"collapse={metrics['mode_collapse_rate']:.3f}, preds={metrics['prediction_counts']}"
        )
        return control


## Cell 6 — GRPO Configuration

In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = GRPOConfig(
    output_dir="ace_grpo_outputs",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_steps=300,
    logging_steps=1,
    save_steps=150,
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_checkpointing=True,
    max_completion_length=64,
    temperature=0.9,
    top_p=0.95,
)

training_args

## Cell 7 — Trainer Setup

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[reward_func],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[HeldoutEvalCallback(every_steps=50, n_rollouts=24)],
)

print("Trainer ready.")

## Cell 8 — Training Loop

In [ ]:
baseline_metrics = evaluate_model(model, tokenizer, eval_dataset, n_rollouts=24)
print("Before training:")
print(json.dumps({k: v for k, v in baseline_metrics.items() if k != 'samples'}, indent=2))
print("Sample completions before training:")
for sample in baseline_metrics["samples"]:
    print("-" * 80)
    print(sample["completion"])
    print(sample["parsed"], "reward=", round(sample["reward"], 3), "correct=", sample["correct"], "valid=", sample["valid"])

train_result = trainer.train()
print(train_result)
print("Logged reward entries:", len(reward_log))
print("Heldout eval entries:", len(heldout_eval_log))

## Cell 9 — Evaluation

In [ ]:
after_metrics = evaluate_model(model, tokenizer, eval_dataset, n_rollouts=24)

print("After training:")
print(json.dumps({k: v for k, v in after_metrics.items() if k != 'samples'}, indent=2))

print("\nImprovement summary:")
print(f"Reward:   {baseline_metrics['mean_reward']:.3f} -> {after_metrics['mean_reward']:.3f}")
print(f"Accuracy: {baseline_metrics['accuracy']:.3f} -> {after_metrics['accuracy']:.3f}")
print(f"Valid:    {baseline_metrics['valid_rate']:.3f} -> {after_metrics['valid_rate']:.3f}")

print("\nSample completions after training:")
for sample in after_metrics["samples"]:
    print("-" * 80)
    print(sample["completion"])
    print(sample["parsed"], "reward=", round(sample["reward"], 3), "correct=", sample["correct"], "valid=", sample["valid"])

## Cell 10 — Save Model

In [ ]:
SAVE_DIR = "ace_qwen_grpo_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the LoRA adapter and tokenizer only. Do not merge a 4-bit model here.
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved adapter to", SAVE_DIR)

## Cell 11 — Visualization

In [ ]:
reward_steps = list(range(len(reward_log)))
avg_rewards = [row["avg_reward"] for row in reward_log]
accuracies = [row["accuracy"] for row in reward_log]
invalid_rates = [row["invalid_rate"] for row in reward_log]
inference_rewards = [row.get("explicit_inference_reward", 0.0) for row in reward_log]
env_rewards = [row.get("clipped_env_reward", 0.0) for row in reward_log]
bid_penalties = [row.get("bid_hack_penalty", 0.0) for row in reward_log]
heldout_steps = [row["step"] for row in heldout_eval_log]
heldout_accs = [row["accuracy"] for row in heldout_eval_log]
heldout_rewards = [row["mean_reward"] for row in heldout_eval_log]
heldout_collapse = [row["mode_collapse_rate"] for row in heldout_eval_log]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: reward + accuracy over training
axes[0].plot(reward_steps, avg_rewards, label="Avg reward", color="steelblue")
axes[0].plot(reward_steps, accuracies, label="Accuracy", color="seagreen", linestyle="--")
axes[0].plot(reward_steps, inference_rewards, label="Explicit inference reward", color="darkorange", alpha=0.75)
axes[0].plot(reward_steps, env_rewards, label="Clipped env reward", color="purple", alpha=0.55)
if heldout_steps:
    axes[0].plot(heldout_steps, heldout_accs, label="Heldout accuracy", color="black", marker="o", linewidth=2)
    axes[0].plot(heldout_steps, heldout_collapse, label="Heldout collapse rate", color="crimson", marker="x", linewidth=2)
axes[0].axhline(y=0.333, color="gray", linestyle=":", alpha=0.6, label="Random baseline (33%)")
axes[0].set_title("Reward & Accuracy — GRPO Training")
axes[0].set_xlabel("Logged step")
axes[0].set_ylabel("Value")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: before/after eval bar chart on a separate axis
labels = ["Before training", "After training"]
acc_vals = [baseline_metrics["accuracy"], after_metrics["accuracy"]]
rew_vals = [baseline_metrics["mean_reward"], after_metrics["mean_reward"]]
collapse_vals = [baseline_metrics["mode_collapse_rate"], after_metrics["mode_collapse_rate"]]

x = [0, 1]
axes[1].bar(x, acc_vals, width=0.35, alpha=0.75, label="Inference accuracy", color="seagreen")
axes[1].bar([xi + 0.38 for xi in x], rew_vals, width=0.35, alpha=0.75, label="Mean reward", color="steelblue")
axes[1].bar([xi + 0.76 for xi in x], collapse_vals, width=0.35, alpha=0.75, label="Mode collapse rate", color="crimson")
axes[1].set_xticks([0.38, 1.38], labels)
axes[1].set_title("Before vs After — Eval Set")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
print("Saved: training_curves.png")
plt.show()

## Notes

- Training is intentionally small so it finishes on a single Hugging Face / Colab GPU.
- The reward uses the real ACE environment step, plus a small JSON-format bonus/penalty for dense learning signal.
- If improvement is noisy, increase `max_steps` modestly or keep the dataset on `difficulty=\"easy\"` for clearer signal.